# Quantum computing basics with Qiskit

Two small circuits, simulated (no real quantum hardware needed): a Bell state, which
demonstrates entanglement directly in the measurement statistics, and Grover's search
algorithm, which demonstrates quantum speedup on the smallest possible non-trivial case.

## Entanglement: the Bell state

`h(0)` puts qubit 0 into an equal superposition of \(|0\rangle\) and \(|1\rangle\). `cx(0, 1)`
then entangles qubit 1 with it. The result is a state that, measured, always gives **both
qubits the same value** - `00` or `11`, roughly 50/50, and **never** `01` or `10` - even though
each qubit individually looks maximally random.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])

sim = AerSimulator()
result = sim.run(transpile(qc, sim), shots=4096).result()
counts = result.get_counts()
print("measurement counts:", counts)

total = sum(counts.values())
frac_correlated = (counts.get("00", 0) + counts.get("11", 0)) / total
print(f"fraction of shots where both qubits agree: {frac_correlated:.3f} (should be ~1.0)")

## Grover's search algorithm

Classically, finding one marked item among 4 unsorted possibilities takes on average ~2.5
guesses (up to 4 in the worst case). Grover's algorithm finds it in exactly **one** query,
using an oracle (which marks the target with a phase flip) and a diffusion step (which
amplifies the marked state's probability). For a 2-qubit search space this is the special case
where a single iteration gives a 100% success probability.

In [ ]:
# Search for |11> among {|00>, |01>, |10>, |11>}
grover = QuantumCircuit(2, 2)
grover.h([0, 1])   # uniform superposition over all 4 basis states
grover.cz(0, 1)    # oracle: flip the phase of |11> only
grover.h([0, 1])   # diffusion operator (amplitude amplification)...
grover.z([0, 1])
grover.cz(0, 1)
grover.h([0, 1])   # ...done
grover.measure([0, 1], [0, 1])

result = sim.run(transpile(grover, sim), shots=4096).result()
counts = result.get_counts()
print("measurement counts:", counts)
print(f"P(11) = {counts.get('11', 0) / sum(counts.values()):.3f} (should be ~1.0)")

## Next steps

- Change the oracle (`grover.cz(0, 1)`) to mark a different target state and confirm Grover
  finds it too.
- Scale up to 3+ qubits - a single Grover iteration is no longer exactly optimal (you'll need
  roughly $\sqrt{N}$ iterations), a good place to see the quadratic speedup show up numerically.
- Swap `AerSimulator()` for `AerSimulator(noise_model=...)` to see how real hardware noise
  degrades these clean results.